In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed
from snowflake.connector.pandas_tools import write_pandas

In [2]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [3]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [6]:
def push_data(data, table_name, conn):
    """Append a prepared dataframe to Snowflake using the shared upload settings.

    Returns the write_pandas result. Re-running appends the same rows again.
    """
    return write_pandas(
        conn,
        data,
        table_name=table_name,
        auto_create_table=True,
        overwrite=False,
    )


def get_m_month(run_month, month):
    """Return M through M+8 using the existing exact-date matching rules."""
    run_month = pd.to_datetime(run_month)
    month = pd.to_datetime(month)
    labels = pd.Series(None, index=run_month.index, dtype=object)
    labels.loc[month.eq(run_month)] = 'M'
    for offset in range(1, 9):
        labels.loc[month.eq(run_month + MonthEnd(offset))] = f'M+{offset}'
    return labels


realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Push heuristic data

In [ ]:
# df = pd.read_excel('/data/aman_singh/acuuracy_check/All_combination_Nov25_live_with_festivals((Autorecovered-312166393861403186)).xlsb', sheet_name = 'Base')
# df

,key,month_date,pred_prophet,pred_rf,pred_value_prophet,pred_value_rf,channel,asm_area_code,depot_code,parent_material_code,...,RF_Recency NS Heuristic Val,RF_Final Heursitic Val,Remark,Missing,ALL Planning Principle,NON ALL Planning Principle,Skip basis PP,Heuristic > 2x Model,Diff,Comparison
0,BCE1_D231_718589,45991,5.264959,2.7,0.000237,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
1,BCE1_D231_718589,46022,4.547336,3.6,0.000205,0.000162,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
2,BCE1_D231_718589,46053,7.053803,2.7,0.000317,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
3,BCE1_D231_718589,46081,5.662848,1.8,0.000255,0.000081,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,0,0.0,P3M
4,BCE1_D231_718589,46112,0.000000,2.7,0.000000,0.000122,ECOM,BCE1,D231,718589,...,0.000203,0.000203,NaN,0,0,0,0,1,0.0,P3M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCW2_D463_810125,46022,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164366,QCW2_D463_810125,46053,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164367,QCW2_D463_810125,46081,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M
164368,QCW2_D463_810125,46112,NaN,NaN,NaN,NaN,QCOM,QCW2,D463,810125,...,0.000069,0.000092,NaN,1,0,0,0,1,0.0,P3M


In [ ]:
# df['month_date'] = (
#     pd.to_datetime(df['month_date'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )
# df['run_month'] = (
#     pd.to_datetime(df['run_month'], unit='D', origin='1899-12-30')
#       .dt.to_period('M')
#       .dt.to_timestamp()
# )

# df['month_date'] = (
#     pd.to_datetime(df['month_date']) + pd.offsets.MonthEnd(0)
# )
# df['run_month'] = (
#     pd.to_datetime(df['run_month']) + pd.offsets.MonthEnd(0)
# )

In [ ]:
# df['month_date'] = df['month_date'].astype(str)
# df['run_month'] = df['run_month'].astype(str)
# df.columns

Index(['key', 'month_date', 'pred_prophet', 'pred_rf', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'sec_vol_actuals_rum_month_value',
       'pred_best_model', 'pred_value_best_model',
       'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov',
       'run_month', 'M month', 'pred_prophet_70%ile', 'portfolio',
       'qtr_ind_rate', 'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M',
       'LY P6M', 'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2', 'Growth Flag',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'LY', 'LLY', 'LY value', 'LLY value',
       'Sec_Value_in_Cr_lag_1', 'Sec_Value_in_Cr_lag_2',
       'Sec_Value_in_Cr_lag_3', 'ASM', 'Depot', 'PSKU', 'class', '

In [ ]:
# upload_df = df[['channel','portfolio', 'brand_code', 'class',
#         'run_month', 'M month','month_date', 'ASM', 'Depot', 'PSKU','pred_prophet', 'pred_rf','Final Heuristic 2 Vol',
#         'RF_Final Heuristic Vol']].rename(
#             columns = {'brand_code':'brand', 'class':'Brand Class', 'month_date':'month','pred_prophet':'prophet vol',
#                        'pred_rf':'rf_vol', 'Final Heuristic 2 Vol':'prophet heuristic vol',
#                        'RF_Final Heuristic Vol':'rf heuristic vol'}
#         )
# upload_df

,channel,portfolio,brand,Brand Class,run_month,M month,month,ASM,Depot,PSKU,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
# upload_df.columns = upload_df.columns.str.upper()
# upload_df

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.7,5.264959,4.50
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.6,4.547336,4.50
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.7,7.053803,4.50
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.8,5.662848,4.50
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.7,4.500000,4.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164365,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+1,2025-12-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164366,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+2,2026-01-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164367,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+3,2026-02-28,QCW2,D463,810125,NaN,NaN,0.640000,0.64
164368,QCOM,Male Grooming,SW_SGPRF,C,2025-11-30,M+4,2026-03-31,QCW2,D463,810125,NaN,NaN,0.640000,0.64


In [ ]:
# push_data(upload_df, 'TRN_MIL_DF_HEURISTICS_OUTPUT', dev_conn)

### Push shared output

Edit the file mapping and run month below, then run this section once to prepare and push **both GT and MT**. The existing append behavior is retained; rerunning the push cell appends again.

In [48]:
def prepare_shared_output(file_path, channel, run_month):
    df = pd.read_excel(file_path, sheet_name='Base')
    # Validate the file/channel pairing instead of relabelling another channel.
    if not df['Channel'].eq(channel).all():
        raise ValueError(f'Expected only {channel} rows in {file_path}')
    df['run_month'] = pd.Timestamp(run_month)
    df['Month'] = (
        pd.to_datetime(df['Month'], unit='D', origin='1899-12-30')
        .dt.to_period('M').dt.to_timestamp() + MonthEnd(0)
    )
    df['M month'] = get_m_month(df['run_month'], df['Month'])
    df['Month'] = df['Month'].astype(str)
    df['run_month'] = df['run_month'].astype(str)
    upload_df = df[['Channel', 'Portfolio', 'Brand', 'Brand Class',
                    'run_month', 'M month', 'Month', 'ASM', 'Depot', 'PSKU',
                    'Pred Vol (ROUM)']].copy()
    upload_df.columns = upload_df.columns.str.upper()
    upload_df['PORTFOLIO'] = upload_df['PORTFOLIO'].astype(str)
    return upload_df

In [50]:
shared_files = {
    'GT': '/data/aman_singh/acuuracy_check/Stat_Demand_Forecast_GT_04092026_updated 1.xlsb',
    'MT': '/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_8th_sep_2026.xlsb',
}
shared_run_month = '2026-09-30'

shared_outputs = {
    channel: prepare_shared_output(file_path, channel, shared_run_month)
    for channel, file_path in shared_files.items()
}
for channel, output in shared_outputs.items():
    print(f'{channel}: {len(output):,} rows prepared')

GT: 46,916 rows prepared
MT: 29,456 rows prepared


In [52]:
dfs = shared_outputs['MT']
dfs.groupby(['MONTH'])['PRED VOL (ROUM)'].sum()

MONTH
2026-10-31    879071.766681
2026-11-30    921527.430409
2026-12-31    874956.924100
2027-01-31    825558.663443
Name: PRED VOL (ROUM), dtype: float64

In [53]:
shared_upload_results = {}
for channel, output in shared_outputs.items():
    shared_upload_results[channel] = push_data(output, 'TRN_MIL_DF_SHARED', dev_conn)
shared_upload_results

{'GT': (True,
  1,
  46916,
  [('jpaahnlddg/file0.txt',
    'LOADED',
    46916,
    46916,
    1,
    0,
    None,
    None,
    None,
    None)]),
 'MT': (True,
  1,
  29456,
  [('rshoppqakf/file0.txt',
    'LOADED',
    29456,
    29456,
    1,
    0,
    None,
    None,
    None,
    None)])}

### push offtakes to primary

Run this section once for **QCOM and Ecom depot-level primary forecasts**. The portfolio lookup is loaded once and reused. Preparation remains separate from the append cell for inspection.

In [22]:
portfolio_query = """SELECT DISTINCT
    MATERIAL_GROUP_CODE AS BRAND_CODE,
    SAP_PORTFOLIO_NAME AS PORTFOLIO
FROM MST_MATERIAL
WHERE COMPANY_CODE = 'MIL'
  AND LATEST_RECORD_IND = '1'
  AND MATERIAL_GROUP_CODE IS NOT NULL
  AND SAP_PORTFOLIO_NAME IS NOT NULL
ORDER BY SAP_PORTFOLIO_NAME, MATERIAL_GROUP_CODE;"""
brand_md_df = pd.read_sql(portfolio_query, prod_conn)

brand_md_df.columns = brand_md_df.columns.str.lower()
brand_md_df.rename(columns = {'brand_code':'Brand'}, inplace = True)

In [25]:
def prepare_primary_output(file_path, channel, brand_metadata, grain='Depot'):
    if grain not in ('Depot', 'Chain'):
        raise ValueError("grain must be 'Depot' or 'Chain'")
    try:
        df = pd.read_excel(file_path, sheet_name='Base')
    except ValueError:
        df = pd.read_excel(file_path, sheet_name='base')
    df['Run Month'] = pd.to_datetime(df['Run Month'])
    df['Month Date'] = pd.to_datetime(df['Month Date'])
    df['M month'] = get_m_month(df['Run Month'], df['Month Date'])
    df.rename(columns={'Calculated Depot PSKU Primary Vol': 'Calculated Primary Vol'}, inplace=True)
    df = df.merge(brand_metadata, on='Brand', how='left', validate='many_to_one')
    df = df.groupby(['Month Date', grain, 'PSKU', 'Brand', 'portfolio',
                     'M month', 'Run Month'])[['Calculated Primary Vol']].sum().reset_index()
    df.rename(columns={'Month Date': 'Month', 'Run Month': 'run_month',
                       'portfolio': 'Portfolio'}, inplace=True)
    df['Month'] = df['Month'].astype(str)
    df['run_month'] = df['run_month'].astype(str)
    upload_df = df[['Month', grain, 'PSKU', 'Brand', 'Portfolio', 'M month',
                    'run_month', 'Calculated Primary Vol']].copy()
    upload_df['Channel'] = channel
    upload_df.columns = upload_df.columns.str.upper()
    return upload_df

In [26]:
primary_files = {
    'QCOM': '/data/aman_singh/acuuracy_check/Stat_demand_forecast_Qcom_as_on_3rd_September.xlsx',
    'Ecom': '/data/aman_singh/acuuracy_check/Stat_forecast_Ecom_depot_psku_as_on_11th_sep_2026.xlsx',
}
primary_outputs = {
    channel: prepare_primary_output(file_path, channel, brand_md_df)
    for channel, file_path in primary_files.items()
}
for channel, output in primary_outputs.items():
    print(f'{channel}: {len(output):,} rows prepared')

QCOM: 10,846 rows prepared
Ecom: 17,603 rows prepared


In [31]:
primary_upload_results = {}
for channel, output in primary_outputs.items():
    primary_upload_results[channel] = push_data(output, 'TRN_MIL_DF_OFT2PRIM_SHARED', dev_conn)
primary_upload_results

{'QCOM': (True,
  1,
  10846,
  [('xqbglxhzvm/file0.txt',
    'LOADED',
    10846,
    10846,
    1,
    0,
    None,
    None,
    None,
    None)]),
 'Ecom': (True,
  1,
  17603,
  [('kilehrsnqq/file0.txt',
    'LOADED',
    17603,
    17603,
    1,
    0,
    None,
    None,
    None,
    None)])}

### Push chain psku primary

Reuse the same primary preparation function at Chain grain. Add other channel files to this mapping when needed.

In [32]:
chain_primary_files = {
    'Ecom': '/data/aman_singh/acuuracy_check/Stat_forecast_Ecom_chain_psku_as_on_11th_sep_2026.xlsx',
    'QCOM':'/data/aman_singh/acuuracy_check/Stat_demand_forecast_Qcom_as_on_3rd_September.xlsx'
}
chain_primary_outputs = {
    channel: prepare_primary_output(file_path, channel, brand_md_df, grain='Chain')
    for channel, file_path in chain_primary_files.items()
}
for channel, output in chain_primary_outputs.items():
    print(f'{channel}: {len(output):,} rows prepared')

Ecom: 5,896 rows prepared
QCOM: 1,799 rows prepared


In [35]:
chain_primary_outputs['QCOM']['CALCULATED PRIMARY VOL'].sum()

152333.48026492915

In [36]:
chain_upload_results = {}
for channel, output in chain_primary_outputs.items():
    chain_upload_results[channel] = push_data(output, 'TRN_MIL_DF_OFT2PRIM_CPSKU', dev_conn)
chain_upload_results

{'Ecom': (True,
  1,
  5896,
  [('yetdsjhhrs/file0.txt',
    'LOADED',
    5896,
    5896,
    1,
    0,
    None,
    None,
    None,
    None)]),
 'QCOM': (True,
  1,
  1799,
  [('rteapspvqm/file0.txt',
    'LOADED',
    1799,
    1799,
    1,
    0,
    None,
    None,
    None,
    None)])}

### push offtakes data

Configure each channel once below, then prepare both outputs and run the upload loop. QCOM uses `final_heuristic_value`; Ecom uses `final Heuristic Value` from its lowercase `base` sheet. Existing grouping, date formatting, destination columns, and append behavior are retained.

In [41]:
def prepare_offtakes_output(file_path, channel, sheet_name='Base',
                            value_column='final_heuristic_value'):
    if not value_column:
        raise ValueError(f'Set value_column for {channel} before preparing uploads')

    df = pd.read_excel(file_path, sheet_name=sheet_name)
    
    if 'run_month' not in df.columns:
        df.rename(columns={'run_month_x': 'run_month'}, inplace=True)
    keys = ['month_date', 'platform_name', 'parent_material_code', 'brand_code',
            'portfolio', 'run_month', 'M month']
    values = ['pred_prophet', 'pred_rf', value_column]
    missing = [column for column in keys + values if column not in df.columns]
    if missing:
        raise ValueError(f'{channel}: missing columns {missing}')
    upload_df = df.groupby(keys)[values].sum().reset_index()
    upload_df.rename(columns={value_column: 'final_heuristic_prophet_value_2'}, inplace=True)
    upload_df['month_date'] = upload_df['month_date'].astype(str)
    upload_df['run_month'] = upload_df['run_month'].astype(str)
    upload_df['channel'] = channel
    upload_df.columns = upload_df.columns.str.upper()
    return upload_df

In [42]:
offtakes_files = {
    'QCOM': {
        'file_path': '/data/aman_singh/acuuracy_check/heuristic_all_combination_qcom_sep_pred.xlsx',
        'sheet_name': 'Base',
        'value_column': 'final_heuristic_value',
    },
    'Ecom': {
        'file_path': '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_sep_live.xlsx',
        'sheet_name': 'base',
        'value_column': 'final Heuristic Value',
    },
}

# Fail before reading or uploading either channel if a mapping is incomplete.
for channel, config in offtakes_files.items():
    if not config['value_column']:
        raise ValueError(f'Set offtakes_files[{channel!r}]["value_column"]')

offtakes_outputs = {
    channel: prepare_offtakes_output(channel=channel, **config)
    for channel, config in offtakes_files.items()
}
for channel, output in offtakes_outputs.items():
    print(f'{channel}: {len(output):,} rows prepared')

QCOM: 5,455 rows prepared
Ecom: 29,596 rows prepared


In [47]:
offtakes_upload_results = {}
for channel, output in offtakes_outputs.items():
    offtakes_upload_results[channel] = push_data(output, 'TRN_MIL_DF_OFFTAKES_OUTPUT', dev_conn)
offtakes_upload_results

{'QCOM': (True,
  1,
  5455,
  [('gsdljzxngw/file0.txt',
    'LOADED',
    5455,
    5455,
    1,
    0,
    None,
    None,
    None,
    None)]),
 'Ecom': (True,
  1,
  29596,
  [('onrtzvnirr/file0.txt',
    'LOADED',
    29596,
    29596,
    1,
    0,
    None,
    None,
    None,
    None)])}